## **Fire Heatmap of South America**
-----
#### SDS 210 - Programming with Spatial Data

*Author: Isabelle Bartholet*

*Date: May 2026*

### **Research Question**
* How has the fire distribution in South America varied over the past five days?
* Are there any patterns identifiable?


### **Content**

1. Load important packages 
2. Call map key via function check_map_key()
3. Fetch the Data via function fetch_data()
4. Convert Data to gdf
5. Create Heatmap


#### **Import Libraries and Packages**

In [ ]:
import requests
import pandas as pd
import geopandas as gpd
import time
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import folium
import contextily
import cmcrameri
import branca.colormap as cm
import numpy as np



from cartopy import crs as ccrs
from geodatasets import get_path


In [2]:
from config import MAP_KEY
from access_mapkey import check_map_key

#### **Access Map Key**

In [3]:
check_map_key(MAP_KEY)

transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object


{'transaction_limit': 5000,
 'current_transactions': 0,
 'transaction_interval': '10 minutes'}

#### **Import Data for the Last Five Days**

In [4]:
from FetchFireData2 import fetch_data
from config import MAP_KEY

df_fire = fetch_data(MAP_KEY)

# check length of df_fire, to see whether it worked correctly
print(f"There are {len(df_fire)} rows in this data frame.")

# quickly check dataframe by using head
df_fire.head(6)


Request successful


 Data download successful! 7817 fires found.
There are 7817 rows in this data frame.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,-10.26649,-38.14394,315.39,0.41,0.45,2026-05-13,332,N20,VIIRS,n,2.0NRT,284.57,1.46,N
1,-8.54213,-45.98396,325.67,0.72,0.76,2026-05-13,332,N20,VIIRS,n,2.0NRT,287.36,3.09,N
2,-8.54070,-45.99136,299.63,0.72,0.76,2026-05-13,332,N20,VIIRS,n,2.0NRT,286.78,3.09,N
3,-8.54055,-45.98581,330.77,0.72,0.76,2026-05-13,332,N20,VIIRS,n,2.0NRT,287.06,6.33,N
4,-3.59582,-38.86000,317.12,0.58,0.52,2026-05-13,332,N20,VIIRS,n,2.0NRT,282.58,3.32,N
5,-3.59188,-38.85688,303.36,0.58,0.52,2026-05-13,332,N20,VIIRS,n,2.0NRT,282.14,2.24,N


The data is stored in a dataframe called df_fire.

#### **Convert Fire Dataframe into GDF and Check for NAs**


In [5]:
# create gdf 
gdf_fire = gpd.GeoDataFrame(
    df_fire, 
    geometry=gpd.points_from_xy(
        df_fire.longitude, df_fire.latitude),
        
        crs="EPSG:4326")

# check, to see if gdf returns the same information as the df above 
gdf_fire.head(6)


# check for NAs
gdf_fire.info()
print(f"There are no NAs in this gdf.") ## noch ändern zu etwas, was mit gdf_fire.info() zusammenhängt


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 7817 entries, 0 to 7816
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   latitude    7817 non-null   float64 
 1   longitude   7817 non-null   float64 
 2   bright_ti4  7817 non-null   float64 
 3   scan        7817 non-null   float64 
 4   track       7817 non-null   float64 
 5   acq_date    7817 non-null   str     
 6   acq_time    7817 non-null   int64   
 7   satellite   7817 non-null   str     
 8   instrument  7817 non-null   str     
 9   confidence  7817 non-null   str     
 10  version     7817 non-null   str     
 11  bright_ti5  7817 non-null   float64 
 12  frp         7817 non-null   float64 
 13  daynight    7817 non-null   str     
 14  geometry    7817 non-null   geometry
dtypes: float64(7), geometry(1), int64(1), str(6)
memory usage: 1.1 MB
There are no NAs in this gdf.


#### **Heatmap**

Time animated heatmap including a scale and zoom control.

In [46]:
from folium.plugins import HeatMapWithTime


## convert dates into a specialized time-aware objects (datetime64)
gdf_fire["acq_date"] = pd.to_datetime(gdf_fire["acq_date"])


## sort days
unique_days = sorted(gdf_fire["acq_date"].dt.date.unique())

# gdf_fire["acq_date"]: access "aqc_date" column in gdf_fire -> returns Pandas-Series
# dt.: to access time/date values (from datetime)
# .date: removes hours from timeobject
# .unique: takes unique values (maybe unnecessary?)
# sorted(): sorts it in chronological order -> result is a sorted list




## create heatmap data, so HeatMapWithTime can use / work with it

data = [] #create empty list; here all the days will be saved
max_frp = gdf_fire["frp"].max()


for day in unique_days:
    subset = gdf_fire[gdf_fire["acq_date"].dt.date == day] ## ist das .dt.date nochmals nötig?
    day_data = [
        [float(row.geometry.y), float(row.geometry.x), 
         float(row["frp"]/ max_frp)] #float() to create nromal Python float values, currently they are np.float64 dataypes (see .info() above)
         for _, row in subset.iterrows()
    ]

    data.append(day_data)

## create time index
time_index = [str(day) for day in unique_days]


## initialize basemap centered on South America, add a zoom control limit
min_lon, max_lon = -81.5, -35.0
min_lat, max_lat = -64.0, 15.5

base_map = folium.Map(
    max_bounds = True,
    location = [8, -69],
    zoom_start = 3,
    tiles = "CartoDB DarkMatter", #or better with CartoDB Positron
    control_scale = True,
    min_lat = min_lat,
    max_lat = max_lat,
    min_lon = min_lon,
    max_lon = max_lon

)

## heatmap
# 0-1 as threshold values for pixel intensity
HeatMapWithTime(
    data,
    index=time_index,
    auto_play = True,
    radius = 10,
    min_opacity= 0.3,
    gradient = {
        0.0: "#0000ff", # isolated value
        0.5: "#f9e107", #medium density
        1.0: "#ff0101", # high density
    }
).add_to(base_map)


## html legend
legend_html = """
<div style = "position: fixed;
bottom: 50px; 
right: 50px;
width: 170px; 
height: 100px;
background-color: lightgrey;
z-index:9999;
font-size:14px;
padding: 10px;
border: 2px solid grey;
">

<b>Fire Density</b><br>
<i style="background:#0000ff;width:20px;height:10px;float:left;margin-right:8px;"></i> Low Density<br>

<i style="background:#f9e107;width:20px;height:10px;float:left;margin-right:8px;"></i> Medium Density<br>

<i style="background:#ff0101;width:20px;height:10px;float:left;margin-right:8px;"></i> High Density

</div>
"""

## add legend to the map
base_map.get_root().html.add_child(folium.Element(legend_html))


## save map because I cannot open it in VS Code directly
base_map.save("Time_Animated_Heatmap.html")

base_map


In [ ]:
from cartopy import crs as ccrs
from geodatasets import get_path
import branca.colormap as cm
import numpy as np
